In [3]:

# Decision Tree using ID3 Algorithm
# Machine Learning Lab Program

import pandas as pd
import math
from collections import Counter

# Step 1: Create the training dataset
data = {
    'Outlook': ['Sunny', 'Sunny', 'Overcast', 'Rain',
                'Rain', 'Rain', 'Overcast', 'Sunny',
                'Sunny', 'Rain', 'Sunny', 'Overcast',
                'Overcast', 'Rain'],
    'Temperature': ['Hot', 'Hot', 'Hot', 'Mild',
                    'Cool', 'Cool', 'Cool', 'Mild',
                    'Cool', 'Mild', 'Mild', 'Mild',
                    'Hot', 'Mild'],
    'Humidity': ['High', 'High', 'High', 'High',
                 'Normal', 'Normal', 'Normal', 'High',
                 'Normal', 'Normal', 'Normal', 'High',
                 'Normal', 'High'],
    'Wind': ['Weak', 'Strong', 'Weak', 'Weak',
             'Weak', 'Strong', 'Strong', 'Weak',
             'Weak', 'Weak', 'Strong', 'Strong',
             'Weak', 'Strong'],
    'Play': ['No', 'No', 'Yes', 'Yes',
             'Yes', 'No', 'Yes', 'No',
             'Yes', 'Yes', 'Yes', 'Yes',
             'Yes', 'No']
}

df = pd.DataFrame(data)

print("Training Dataset:")
print(df.to_string(index=False))

# Step 2: Calculate entropy
def entropy(target):
    counts = Counter(target)
    total = len(target)
    result = 0

    for count in counts.values():
        p = count / total
        result -= p * math.log2(p)

    return result

# Step 3: Calculate information gain
def information_gain(data, attribute, target):
    total_entropy = entropy(data[target])
    weighted_entropy = 0

    for value in data[attribute].unique():
        subset = data[data[attribute] == value]
        weighted_entropy += (
            len(subset) / len(data)
        ) * entropy(subset[target])

    return total_entropy - weighted_entropy

# Step 4: Build the ID3 decision tree
def id3(data, attributes, target):
    labels = data[target]

    # If all examples have the same class
    if len(labels.unique()) == 1:
        return labels.iloc[0]

    # If no attributes remain
    if not attributes:
        return labels.mode()[0]

    # Select attribute with highest information gain
    gains = {
        attr: information_gain(data, attr, target)
        for attr in attributes
    }

    best_attribute = max(gains, key=gains.get)

    tree = {best_attribute: {}}
    remaining = [a for a in attributes
                 if a != best_attribute]

    for value in data[best_attribute].unique():
        subset = data[data[best_attribute] == value]

        if subset.empty:
            tree[best_attribute][value] = labels.mode()[0]
        else:
            tree[best_attribute][value] = id3(
                subset, remaining, target
            )

    return tree

# Step 5: Display information gain
attributes = list(df.columns[:-1])

print("\nInformation Gain:")
for attr in attributes:
    print(f"{attr}: {information_gain(df, attr, 'Play'):.4f}")

# Build the decision tree
tree = id3(df, attributes, 'Play')

print("\nDecision Tree:")
print(tree)

# Step 6: Classify a new sample
def classify(tree, sample):
    if not isinstance(tree, dict):
        return tree

    attribute = next(iter(tree))
    value = sample.get(attribute)

    if value not in tree[attribute]:
        return "Unknown"

    return classify(tree[attribute][value], sample)

new_sample = {
    'Outlook': 'Sunny',
    'Temperature': 'Cool',
    'Humidity': 'High',
    'Wind': 'Strong'
}

prediction = classify(tree, new_sample)

print("\nNew Sample:")
print(new_sample)

print("Predicted Class:", prediction)


Training Dataset:
 Outlook Temperature Humidity   Wind Play
   Sunny         Hot     High   Weak   No
   Sunny         Hot     High Strong   No
Overcast         Hot     High   Weak  Yes
    Rain        Mild     High   Weak  Yes
    Rain        Cool   Normal   Weak  Yes
    Rain        Cool   Normal Strong   No
Overcast        Cool   Normal Strong  Yes
   Sunny        Mild     High   Weak   No
   Sunny        Cool   Normal   Weak  Yes
    Rain        Mild   Normal   Weak  Yes
   Sunny        Mild   Normal Strong  Yes
Overcast        Mild     High Strong  Yes
Overcast         Hot   Normal   Weak  Yes
    Rain        Mild     High Strong   No

Information Gain:
Outlook: 0.2467
Temperature: 0.0292
Humidity: 0.1518
Wind: 0.0481

Decision Tree:
{'Outlook': {'Sunny': {'Humidity': {'High': 'No', 'Normal': 'Yes'}}, 'Overcast': 'Yes', 'Rain': {'Wind': {'Weak': 'Yes', 'Strong': 'No'}}}}

New Sample:
{'Outlook': 'Sunny', 'Temperature': 'Cool', 'Humidity': 'High', 'Wind': 'Strong'}
Predicted Class: